[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C47_RecSys_Ranking_Course/04_learning_to_rank/04_learning_to_rank.ipynb)

# 04 · 排序学习（用 numpy 从零）

目标：从零实现 **BPR / pairwise 排序** 与 **nDCG/MAP 评估**——理解 **pointwise vs pairwise**、**BPR 损失与梯度**（对拍数值梯度）、**nDCG 的位置折损与归一化**，验证 pairwise 排序提升 nDCG。

路线：数据 → nDCG/DCG/IDCG(对拍) → MAP → BPR 损失+梯度(对拍) → BPR 训练看 nDCG 上升 → pairwise vs pointwise → ✏️ 练习 → 📖 答案 → 🧪 真实 MovieLens 排序胶囊。

> 核心心智：**排序是相对的→损失也该相对（pairwise: 正例分>负例分）；真正想要的 nDCG 不可导→用 pairwise 当可导代理。**

## 0 · 数据：隐式正例对 + 留一评估

复用加载器，取高分交互当正例，每用户留一个做测试（截断加速）。

In [ ]:
import numpy as np

def load_movielens_or_synth(n_users=200, n_items=300, rank=8, seed=0, verbose=True, cap_users=None, cap_items=None):
    import os
    data = None
    for path in ['ml-100k/u.data', 'u.data', os.path.expanduser('~/ml-100k/u.data')]:
        if os.path.exists(path):
            data = np.loadtxt(path, dtype=np.int64)[:, :3].astype(float); data[:,0]-=1; data[:,1]-=1; break
    if data is None:
        try:
            import urllib.request
            raw = urllib.request.urlopen('https://files.grouplens.org/datasets/movielens/ml-100k/u.data', timeout=5).read().decode()
            rows = [list(map(int, ln.split('\t')[:3])) for ln in raw.strip().split('\n')]
            data = np.array(rows, dtype=float); data[:,0]-=1; data[:,1]-=1
        except Exception as e:
            if verbose: print(f'回退合成（{type(e).__name__}）')
            rng = np.random.default_rng(seed)
            P = rng.standard_normal((n_users, rank))*0.5; Q = rng.standard_normal((n_items, rank))*0.5
            bu = rng.standard_normal(n_users)*0.3; bi = rng.standard_normal(n_items)*0.5; rows=[]
            for u in range(n_users):
                for i in rng.choice(n_items, size=rng.integers(20,60), replace=False):
                    r = 3.5 + bu[u] + bi[i] + P[u]@Q[i] + rng.standard_normal()*0.3
                    rows.append([u, i, float(np.clip(np.round(r*2)/2, 1, 5))])
            data = np.array(rows, dtype=float)
    if cap_users or cap_items:
        cu = cap_users or int(data[:,0].max())+1; ci = cap_items or int(data[:,1].max())+1
        data = data[(data[:,0]<cu)&(data[:,1]<ci)]; nu, ni = cu, ci
    else:
        nu, ni = int(data[:,0].max())+1, int(data[:,1].max())+1
    if verbose: print(f'{len(data)} 评分, {nu} 用户, {ni} 物品')
    return data, nu, ni

ratings, n_users, n_items = load_movielens_or_synth(seed=0, cap_users=300, cap_items=600)
pos = ratings[ratings[:,2]>=4.0][:,:2].astype(int)
rng = np.random.default_rng(0)
test_pos = {}; mask = np.ones(len(pos), bool)
for u in np.unique(pos[:,0]):
    idx = np.where(pos[:,0]==u)[0]
    if len(idx) >= 2:
        h = rng.choice(idx); test_pos[int(u)] = int(pos[h,1]); mask[h] = False
train_pos = pos[mask]
user_pos = {}
for u, i in train_pos: user_pos.setdefault(int(u), set()).add(int(i))
print(f'训练正例 {len(train_pos)}, 测试用户 {len(test_pos)}')
assert len(train_pos) > 1000
print('✅ 数据就绪')

## 1 · nDCG：位置折损 + 归一化（对拍）

$$\text{DCG}@k=\sum_{i=1}^k\frac{2^{rel_i}-1}{\log_2(i+1)},\quad \text{nDCG}@k=\frac{\text{DCG}@k}{\text{IDCG}@k}$$
从零实现，验证：**完美排序 nDCG=1**、**头部排错比尾部排错掉得更多**。

In [ ]:
def dcg_at_k(rels, k):
    '''rels: 按预测排序后各位置的相关性列表。'''
    rels = np.asarray(rels[:k], dtype=float)
    discounts = np.log2(np.arange(2, len(rels)+2))      # log2(i+1), i从1开始
    return float(np.sum((2**rels - 1) / discounts))

def ndcg_at_k(ranked_rels, k):
    '''ranked_rels: 按预测分降序排列后，各位置的真实相关性。'''
    dcg = dcg_at_k(ranked_rels, k)
    ideal = dcg_at_k(sorted(ranked_rels, reverse=True), k)  # IDCG: 相关性降序
    return dcg / ideal if ideal > 0 else 0.0

# 完美排序：相关的(1)全在前
perfect = [1, 1, 1, 0, 0, 0]
assert abs(ndcg_at_k(perfect, 6) - 1.0) < 1e-9, '完美排序 nDCG 应=1'
# 头部排错 vs 尾部排错
head_wrong = [0, 1, 1, 1, 0, 0]    # 第1位本该相关却不相关
tail_wrong = [1, 1, 0, 1, 0, 0]    # 错排发生在更靠后
ndcg_head = ndcg_at_k(head_wrong, 6)
ndcg_tail = ndcg_at_k(tail_wrong, 6)
print(f'完美排序 nDCG@6 = {ndcg_at_k(perfect,6):.4f}')
print(f'头部排错 nDCG@6 = {ndcg_head:.4f}')
print(f'尾部排错 nDCG@6 = {ndcg_tail:.4f}')
assert ndcg_head < ndcg_tail, '头部排错应比尾部排错掉得更多（位置折损）'
print('✅ nDCG 正确：完美=1，且头部错排惩罚 > 尾部错排（这正是位置折损的意义）')

## 2 · MAP（平均精度均值）

Average Precision = 在每个**命中位置**算 precision 再对命中数平均；MAP = 对所有用户的 AP 平均。
它强调「相关物品整体靠前」，是 nDCG 之外的另一常用排序指标。

In [ ]:
def average_precision(ranked_items, relevant_set):
    '''ranked_items: 预测排序的物品 id 列表; relevant_set: 相关物品集合。'''
    if not relevant_set:
        return 0.0
    hits = 0; sum_prec = 0.0
    for rank, it in enumerate(ranked_items, start=1):
        if it in relevant_set:
            hits += 1
            sum_prec += hits / rank                     # 命中处的 precision
    return sum_prec / len(relevant_set)

# 相关物品都在最前 -> AP=1
assert abs(average_precision([2,5,9,0,1], {2,5,9}) - 1.0) < 1e-9
# 相关物品分散 -> AP<1
ap_spread = average_precision([2,0,5,1,9], {2,5,9})    # 命中在位置1,3,5
# precision@命中 = 1/1, 2/3, 3/5 -> AP = (1+0.667+0.6)/3
assert abs(ap_spread - (1 + 2/3 + 3/5)/3) < 1e-9
print(f'完美排序 AP = {average_precision([2,5,9,0,1], {2,5,9}):.4f}')
print(f'分散排序 AP = {ap_spread:.4f}')
print('✅ MAP/AP 正确：相关物品越靠前 AP 越高')

## 3 · BPR 损失与梯度（对拍数值梯度）

打分用 MF：$s(u,i)=p_u\cdot q_i$。三元组 $(u,i,j)$，分差 $x=s(u,i)-s(u,j)$，损失 $-\ln\sigma(x)+\lambda\|\cdot\|^2$。
梯度权重 $(1-\sigma(x))$（排错越多权重越大）。实现损失+梯度，**用数值梯度对拍**。

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def init_bpr(n_users, n_items, k=16, seed=0):
    rng = np.random.default_rng(seed)
    return {'P': rng.standard_normal((n_users,k))*0.1, 'Q': rng.standard_normal((n_items,k))*0.1, 'k':k}

def bpr_point_loss(m, u, i, j, lam):
    x = m['P'][u]@m['Q'][i] - m['P'][u]@m['Q'][j]
    return -np.log(sigmoid(x)+1e-12) + lam*(m['P'][u]@m['P'][u] + m['Q'][i]@m['Q'][i] + m['Q'][j]@m['Q'][j])

def bpr_grad(m, u, i, j, lam):
    x = m['P'][u]@m['Q'][i] - m['P'][u]@m['Q'][j]
    c = 1.0 - sigmoid(x)                                # (1-σ(x)) 难例权重
    g_pu = -c*(m['Q'][i]-m['Q'][j]) + 2*lam*m['P'][u]
    g_qi = -c*m['P'][u] + 2*lam*m['Q'][i]
    g_qj = -c*(-m['P'][u]) + 2*lam*m['Q'][j]
    return g_pu, g_qi, g_qj

m = init_bpr(n_users, n_items, k=8, seed=2)
u, i, j, lam = 3, 10, 20, 0.01
g_pu, g_qi, g_qj = bpr_grad(m, u, i, j, lam)
eps = 1e-6
m['P'][u,0]+=eps; lp=bpr_point_loss(m,u,i,j,lam); m['P'][u,0]-=2*eps; lm=bpr_point_loss(m,u,i,j,lam); m['P'][u,0]+=eps
num_pu0 = (lp-lm)/(2*eps)
m['Q'][i,0]+=eps; lp=bpr_point_loss(m,u,i,j,lam); m['Q'][i,0]-=2*eps; lm=bpr_point_loss(m,u,i,j,lam); m['Q'][i,0]+=eps
num_qi0 = (lp-lm)/(2*eps)
print(f'p_u[0]: 解析={g_pu[0]:.6f} 数值={num_pu0:.6f}')
print(f'q_i[0]: 解析={g_qi[0]:.6f} 数值={num_qi0:.6f}')
assert abs(g_pu[0]-num_pu0) < 1e-4 and abs(g_qi[0]-num_qi0) < 1e-4
print('✅ BPR 损失与梯度正确（对拍数值梯度通过）')

## 4 · BPR 训练：看 nDCG 上升

每步抽 (u, 正例 i, 随机负例 j)，SGD 更新。训练若正确，**留一 nDCG@10 应明显上升**。

In [ ]:
def eval_ndcg(m, test_pos, user_pos, n_items, k=10):
    '''留一 nDCG@k：对每个测试用户给全部物品打分，看留出正例的位置。'''
    total = 0.0; cnt = 0
    for u, held in test_pos.items():
        scores = m['P'][u] @ m['Q'].T
        for s in user_pos.get(u, ()): scores[s] = -np.inf   # 屏蔽训练已知正例
        topk = np.argsort(-scores)[:k]
        rels = [1 if it == held else 0 for it in topk]
        total += ndcg_at_k(rels, k); cnt += 1
    return total/cnt

def train_bpr(train_pos, user_pos, n_users, n_items, k=16, lr=0.05, lam=0.01, steps=40000, seed=0, eval_every=None, test_pos=None):
    m = init_bpr(n_users, n_items, k, seed)
    rng = np.random.default_rng(seed)
    N = len(train_pos); curve = []
    for t in range(steps):
        idx = rng.integers(N)
        u, i = int(train_pos[idx,0]), int(train_pos[idx,1])
        j = rng.integers(n_items)
        while j in user_pos.get(u, ()):                  # 负例不能是该用户的正例
            j = rng.integers(n_items)
        g_pu, g_qi, g_qj = bpr_grad(m, u, i, j, lam)
        m['P'][u] -= lr*g_pu; m['Q'][i] -= lr*g_qi; m['Q'][j] -= lr*g_qj
        if eval_every and test_pos and (t+1) % eval_every == 0:
            curve.append(eval_ndcg(m, test_pos, user_pos, n_items, 10))
    return m, curve

ndcg_init = eval_ndcg(init_bpr(n_users, n_items, 16), test_pos, user_pos, n_items, 10)
m, curve = train_bpr(train_pos, user_pos, n_users, n_items, k=16, lr=0.05, lam=0.01,
                     steps=60000, eval_every=20000, test_pos=test_pos)
ndcg_final = eval_ndcg(m, test_pos, user_pos, n_items, 10)
print(f'nDCG@10: 随机初始={ndcg_init:.4f} -> BPR训练后={ndcg_final:.4f}')
print(f'训练中 nDCG 曲线: {[round(x,4) for x in curve]}')
assert ndcg_final > ndcg_init, 'BPR 训练应提升 nDCG'
assert ndcg_final > 10/n_items, 'nDCG 应远超随机水平'
print('✅ BPR 训练成功：nDCG@10 明显上升（pairwise 损失确实在优化排序）')

## 5 · pairwise vs pointwise：谁更会排序？

pointwise：把正例当 1、随机负例当 0，做**二分类**（logistic/BCE 回归 $\sigma(p\cdot q)$）。
BCE 梯度 $(\hat y-y)\cdot\text{feat}$ 健康（不像 MSE-on-sigmoid 会梯度消失）。对比 pointwise 与 BPR(pairwise) 的 nDCG。

In [ ]:
def train_pointwise(train_pos, user_pos, n_users, n_items, k=16, lr=0.05, lam=0.01, steps=60000, seed=0):
    '''pointwise: 正例标签1、随机负例标签0，logistic(BCE) 二分类 σ(p·q)。'''
    m = init_bpr(n_users, n_items, k, seed); rng = np.random.default_rng(seed); N = len(train_pos)
    for t in range(steps):
        idx = rng.integers(N); u, i = int(train_pos[idx,0]), int(train_pos[idx,1])
        if t % 2 == 0:
            item, y = i, 1.0                              # 正例
        else:
            item = rng.integers(n_items)                 # 随机负例
            while item in user_pos.get(u, ()): item = rng.integers(n_items)
            y = 0.0
        pred = sigmoid(m['P'][u]@m['Q'][item])
        err = pred - y                                   # BCE 对 logit 的梯度因子(健康,不消失)
        g_pu = err*m['Q'][item] + lam*m['P'][u]
        g_qi = err*m['P'][u] + lam*m['Q'][item]
        m['P'][u] -= lr*g_pu; m['Q'][item] -= lr*g_qi
    return m

m_point = train_pointwise(train_pos, user_pos, n_users, n_items, lr=0.05, steps=80000)
ndcg_point = eval_ndcg(m_point, test_pos, user_pos, n_items, 10)
ndcg_pair = eval_ndcg(m, test_pos, user_pos, n_items, 10)
print(f'nDCG@10: pointwise(BCE)={ndcg_point:.4f}, pairwise(BPR)={ndcg_pair:.4f}')
assert ndcg_point > 10/n_items, 'pointwise(BCE) 也应学到排序(超随机)'
print('✅ 两种范式都学到了排序；pairwise 直接优化相对顺序，常在排序指标上更优或相当。')
print('   关键认知：pointwise 优化「分准」、pairwise 优化「顺序对」——后者天生为排序设计。')

## 6 · softmax 损失：从 pairwise 走向 listwise

BPR 一次只比 1 个负例（pairwise）。**softmax 损失**一次比**一批负例**：$-\log\frac{e^{s_i}}{e^{s_i}+\sum_j e^{s_j}}$——
这更接近 listwise（一次考虑多个候选的相对关系），也正是模块 03 双塔的训练损失。负例多时它通常比 BPR 收敛更好。

对比 BPR（1 负例）与 softmax（多负例）训练后的 nDCG。

In [ ]:
def train_softmax(train_pos, user_pos, n_users, n_items, k=16, lr=0.05, lam=0.01, steps=60000, n_neg=10, seed=0):
    '''sampled-softmax: 每步 1 正例 + n_neg 负例，做多分类(正例在第0位)。'''
    m = init_bpr(n_users, n_items, k, seed); rng = np.random.default_rng(seed); N = len(train_pos)
    for t in range(steps):
        idx = rng.integers(N); u, i = int(train_pos[idx,0]), int(train_pos[idx,1])
        negs = rng.integers(0, n_items, size=n_neg)
        cand = np.concatenate([[i], negs])               # 候选: 正例 + 负例
        logits = m['Q'][cand] @ m['P'][u]
        pr = np.exp(logits - logits.max()); pr /= pr.sum()
        dlog = pr.copy(); dlog[0] -= 1                    # 标签=第0个(正例)
        # 梯度回传
        gp = (dlog[:,None] * m['Q'][cand]).sum(axis=0) + lam*m['P'][u]
        m['P'][u] -= lr*gp
        for idx2, c2 in enumerate(cand):
            m['Q'][c2] -= lr*(dlog[idx2]*m['P'][u] + lam*m['Q'][c2])
    return m

m_sm = train_softmax(train_pos, user_pos, n_users, n_items, steps=60000, n_neg=10)
ndcg_sm = eval_ndcg(m_sm, test_pos, user_pos, n_items, 10)
print(f'nDCG@10: BPR(1负例)={eval_ndcg(m, test_pos, user_pos, n_items, 10):.4f}, softmax(10负例)={ndcg_sm:.4f}')
assert ndcg_sm > 10/n_items, 'softmax 损失也应学到排序'
print('✅ softmax 损失(多负例)也优化排序；它和 BPR 是「一次比多个 vs 比一个」的关系（接模块03双塔）')

---
## ✏️ 练习 1：pairwise 0/1 排序准确率

一个直观的 pairwise 指标：随机抽很多 (u, 正例 i, 负例 j)，模型给正例打分**高于**负例的比例。
实现 `pairwise_accuracy`，这其实是 **AUC** 的采样估计。

In [ ]:
def pairwise_accuracy(m, train_pos, user_pos, n_items, n_samples=5000, seed=1):
    '''采样 (u,i,j)，返回 s(u,i)>s(u,j) 的比例（≈AUC）。'''
    # TODO:
    #  1) rng；循环 n_samples 次
    #  2) 随机抽一个训练正例 (u,i)，随机负例 j (不在 user_pos[u])
    #  3) 统计 s(u,i)=P[u]@Q[i] > s(u,j) 的比例
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
acc_trained = pairwise_accuracy(m, train_pos, user_pos, n_items)
acc_random = pairwise_accuracy(init_bpr(n_users, n_items, 16), train_pos, user_pos, n_items)
print(f'pairwise 准确率(≈AUC): 随机={acc_random:.3f}, BPR训练后={acc_trained:.3f}')
assert 0 <= acc_trained <= 1 and 0 <= acc_random <= 1
assert abs(acc_random - 0.5) < 0.1, '随机模型 AUC 应≈0.5'
assert acc_trained > 0.7, 'BPR 训练后 AUC 应明显高于 0.5'
print('✅ 练习 1 通过：pairwise 准确率=AUC 采样估计（随机≈0.5，训练后远超）')

## ✏️ 练习 2：带 nDCG 权重的 LambdaRank 梯度因子

LambdaRank 把 pairwise 梯度乘以「交换 i,j 的 nDCG 变化」$|\Delta\text{nDCG}|$。
实现 `delta_ndcg_swap`：给定两个位置的相关性和它们的排名，算交换这两个位置导致的 |nDCG 变化|。

In [ ]:
def delta_ndcg_swap(rel_i, rel_j, rank_i, rank_j, idcg):
    '''交换排名 rank_i, rank_j 两个位置(相关性 rel_i, rel_j)的 |nDCG 变化|。rank 从1开始。'''
    # TODO:
    #  gain_i = (2**rel_i - 1); gain_j = (2**rel_j - 1)
    #  disc(r) = 1/log2(r+1)
    #  交换前 DCG 贡献 = gain_i*disc(rank_i) + gain_j*disc(rank_j)
    #  交换后          = gain_i*disc(rank_j) + gain_j*disc(rank_i)
    #  返回 |(后-前)| / idcg
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
idcg = dcg_at_k([1,1,1], 3)                              # 假设3个相关物品的 IDCG
# 交换一个相关(1)和一个不相关(0)：头部交换影响 > 尾部交换
d_head = delta_ndcg_swap(1, 0, 1, 2, idcg)               # 位置1,2 交换
d_tail = delta_ndcg_swap(1, 0, 9, 10, idcg)             # 位置9,10 交换
print(f'|ΔnDCG| 头部(位置1,2)交换 = {d_head:.4f}')
print(f'|ΔnDCG| 尾部(位置9,10)交换 = {d_tail:.4f}')
assert d_head > d_tail, '头部交换的 nDCG 影响应远大于尾部（LambdaRank 据此加权）'
assert d_head > 0
# 交换两个相同相关性的物品：nDCG 不变
assert abs(delta_ndcg_swap(1, 1, 1, 2, idcg)) < 1e-12, '相同相关性交换 nDCG 不变'
print('✅ 练习 2 通过：|ΔnDCG| 正确——头部错排影响远大于尾部，这就是 LambdaRank 聚焦头部的原理')

## ✏️ 练习 3：完整 nDCG@K 评估流程

把打分、排序、屏蔽、nDCG 串起来。实现 `score_and_ndcg`：
给一个用户、模型、留出的正例集合，返回该用户的 nDCG@K（屏蔽训练已知正例）。

In [ ]:
def score_and_ndcg(m, u, held_items, train_seen, n_items, k=10):
    '''held_items: 该用户留出的相关物品集合; train_seen: 训练已交互(要屏蔽)。返回 nDCG@k。'''
    # TODO:
    #  1) scores = m['P'][u] @ m['Q'].T
    #  2) 屏蔽 train_seen（设 -inf）
    #  3) topk = argsort 降序 取 k
    #  4) rels = [1 if it in held_items else 0 for it in topk]
    #  5) 返回 ndcg_at_k(rels, k)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 用训练好的模型，对一个测试用户算 nDCG（held 为单个留出正例）
u_test = list(test_pos.keys())[0]
held = {test_pos[u_test]}
nd = score_and_ndcg(m, u_test, held, user_pos.get(u_test, set()), n_items, k=10)
assert 0 <= nd <= 1
# 与 worked 的 eval_ndcg 在该用户上一致
scores = m['P'][u_test] @ m['Q'].T
for s in user_pos.get(u_test, ()): scores[s] = -np.inf
topk = np.argsort(-scores)[:10]
rels = [1 if it == test_pos[u_test] else 0 for it in topk]
assert abs(nd - ndcg_at_k(rels, 10)) < 1e-9, '应与手算一致'
# 群体 nDCG
group = np.mean([score_and_ndcg(m, u, {test_pos[u]}, user_pos.get(u,set()), n_items, 10) for u in list(test_pos)[:50]])
print(f'用户 {u_test} nDCG@10 = {nd:.4f} | 前50用户平均 nDCG@10 = {group:.4f}')
assert group > 10/n_items, '群体 nDCG 应超随机'
print('✅ 练习 3 通过：完整 nDCG@K 评估流程正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def pairwise_accuracy(m, train_pos, user_pos, n_items, n_samples=5000, seed=1):
    rng = np.random.default_rng(seed); N = len(train_pos); correct = 0
    for _ in range(n_samples):
        idx = rng.integers(N); u, i = int(train_pos[idx,0]), int(train_pos[idx,1])
        j = rng.integers(n_items)
        while j in user_pos.get(u, ()): j = rng.integers(n_items)
        correct += int(m['P'][u]@m['Q'][i] > m['P'][u]@m['Q'][j])
    return correct / n_samples

# 练习 2 参考答案
def delta_ndcg_swap(rel_i, rel_j, rank_i, rank_j, idcg):
    gi, gj = 2**rel_i - 1, 2**rel_j - 1
    disc = lambda r: 1.0/np.log2(r+1)
    before = gi*disc(rank_i) + gj*disc(rank_j)
    after  = gi*disc(rank_j) + gj*disc(rank_i)
    return abs(after - before) / idcg if idcg > 0 else 0.0

# 练习 3 参考答案
def score_and_ndcg(m, u, held_items, train_seen, n_items, k=10):
    scores = m['P'][u] @ m['Q'].T
    for s in train_seen: scores[s] = -np.inf
    topk = np.argsort(-scores)[:k]
    rels = [1 if it in held_items else 0 for it in topk]
    return ndcg_at_k(rels, k)
print('参考答案已载入')

---
## 🧪 真实数据胶囊：BPR vs 流行度的 nDCG，以及多指标对比

在真实/合成 MovieLens 上，用多个指标（nDCG@10、MAP、Recall@10）评估 BPR 模型，并和流行度基线对比。
**重点**：同一个排序，不同指标可能给出不同的相对结论——这正是工业上要同时报告多指标的原因。

**TODO**：补全 `eval_all_metrics`，对所有测试用户计算 (nDCG@10, MAP, Recall@10) 三个指标的均值。

In [ ]:
def eval_all_metrics(m, test_pos, user_pos, n_items, k=10):
    '''返回 dict: 平均 nDCG@k, MAP, Recall@k。每用户留出1个正例。'''
    ndcgs, aps, recalls = [], [], []
    for u, held in test_pos.items():
        scores = m['P'][u] @ m['Q'].T
        for s in user_pos.get(u, ()): scores[s] = -np.inf
        ranked = np.argsort(-scores)
        topk = ranked[:k]
        # TODO: 计算这个用户的 nDCG@k, AP, Recall@k（held 是单个正例）
        #   rels_k = [1 if it==held else 0 for it in topk]
        #   ndcgs.append(ndcg_at_k(rels_k, k))
        #   aps.append(average_precision(ranked.tolist(), {held}))
        #   recalls.append(1.0 if held in topk else 0.0)
        raise NotImplementedError
    return {'nDCG@%d'%k: np.mean(ndcgs), 'MAP': np.mean(aps), 'Recall@%d'%k: np.mean(recalls)}

In [ ]:
# 自测（胶囊）
metrics_bpr = eval_all_metrics(m, test_pos, user_pos, n_items, k=10)
print('BPR 模型多指标:', {kk: round(vv,4) for kk,vv in metrics_bpr.items()})
# 流行度基线（构造一个「所有用户都用流行度分」的伪模型不方便，直接算）
counts = np.bincount(train_pos[:,1], minlength=n_items).astype(float)
pop_ndcg = []
for u, held in test_pos.items():
    sc = counts.copy()
    for s in user_pos.get(u, ()): sc[s] = -np.inf
    topk = np.argsort(-sc)[:10]
    pop_ndcg.append(ndcg_at_k([1 if it==held else 0 for it in topk], 10))
print(f'流行度基线 nDCG@10 = {np.mean(pop_ndcg):.4f}')
assert all(0 <= v <= 1 for v in metrics_bpr.values())
assert metrics_bpr['nDCG@10'] > 10/n_items, 'BPR nDCG 应超随机'
print('✅ 胶囊通过：多指标评估完成。工业上必须同时看多个指标 + 强基线，单一指标会误导。')

In [ ]:
# 📖 胶囊参考答案
def eval_all_metrics(m, test_pos, user_pos, n_items, k=10):
    ndcgs, aps, recalls = [], [], []
    for u, held in test_pos.items():
        scores = m['P'][u] @ m['Q'].T
        for s in user_pos.get(u, ()): scores[s] = -np.inf
        ranked = np.argsort(-scores); topk = ranked[:k]
        ndcgs.append(ndcg_at_k([1 if it==held else 0 for it in topk], k))
        aps.append(average_precision(ranked.tolist(), {held}))
        recalls.append(1.0 if held in topk else 0.0)
    return {'nDCG@%d'%k: float(np.mean(ndcgs)), 'MAP': float(np.mean(aps)), 'Recall@%d'%k: float(np.mean(recalls))}

### 小结
- 推荐/搜索的本质是**排对顺序**而非预测准评分；LTR 三范式：**pointwise**（单点）/**pairwise**（物品对）/**listwise**（整列表）。
- **BPR**（=RankNet 思想）是 pairwise 奠基：$-\ln\sigma(s_i-s_j)$，梯度权重 $(1-\sigma)$ 自动聚焦难例。
- 真正想要的 **nDCG 不可导**（排序离散）→ 用 pairwise 当可导代理；**LambdaRank** 把 $|\Delta\text{nDCG}|$ 乘进梯度，聚焦头部。
- **nDCG**（位置折损+归一化）是排序黄金指标；配合 **MAP/MRR/Recall**，工业上**多指标+强基线**一起报。

下一站：**模块 05 · CTR 与序列推荐** —— 精排的模型形态（FM 二阶交叉、DLRM）、序列推荐（SASRec 思想）、位置偏置去偏。